# Week 10 — Support Vector Machines & Kernel Methods
## Notebook 5: Support Vectors — The Borderline Witnesses

| | |
|---|---|
| **Difficulty** | ⭐⭐⭐ (Intermediate) |
| **Estimated Time** | 2 hours |
| **Theme** | Email Spam Classification |
| **Prerequisites** | Notebook 04 (SVM Margin & Dual) |

## Why This Matters

An SVM trained on 100,000 emails may ultimately depend on **fewer than 200 of them**. Understanding *which* emails drive the decision boundary — and why — is the key to understanding SVM efficiency, interpretability, and generalization.

Support vectors are not just a mathematical curiosity. They tell you:
- Which emails are the hardest to classify (the "borderline" cases)
- How complex your model is (more SVs = more complex boundary)
- Which training samples you actually need to store for future predictions
- Where the model is most sensitive to data changes

## Real-World Analogy: The Borderline Witnesses

Imagine a jury trial. The prosecution presents 50 witnesses who clearly saw the defendant commit the crime, and the defense presents 50 witnesses who clearly saw the defendant elsewhere. These 100 people cancel each other out — they don't really determine the verdict.

What determines the verdict? The **3 or 4 borderline witnesses** — the ones who were close to the scene, not totally sure what they saw, whose testimony could swing the jury either way.

**Support vectors are exactly those borderline witnesses in your training data.**

In spam classification:
- An email with 97 spam keywords is *obviously* spam — it's far from the boundary. It doesn't influence the classifier.
- An email with 1 spam keyword is *obviously* legitimate — also far from the boundary.
- But an email with borderline characteristics — maybe 3 spam keywords, sent from a real domain, with an unusual subject — **that's a support vector**. It sits right at the margin, and it's the one that shapes where the decision boundary goes.

Remove all the obvious cases from the courtroom, and the verdict doesn't change. Remove a borderline witness — suddenly the whole case shifts.

## Plain English: What Are Support Vectors?

After training a linear SVM, the decision boundary is a hyperplane: `w·x + b = 0`.

On either side of this boundary, there are two **margin boundaries** ("gutters"):
- `w·x + b = +1` (the positive margin — closest positive examples live here)
- `w·x + b = -1` (the negative margin — closest negative examples live here)

**Support vectors are the training points that lie exactly ON these margin boundaries** (in the hard-margin case). They are the closest points to the decision boundary from each class.

### The Key Insight

The SVM objective function — maximizing the margin — can be written entirely in terms of support vectors. The weight vector `w` is literally a weighted sum of support vectors:

$$\mathbf{w} = \sum_{i \in \text{SVs}} \alpha_i y_i \mathbf{x}_i$$

Non-support vectors have $\alpha_i = 0$ — they contribute **nothing** to `w`.

### Hard-Margin vs Soft-Margin

| Setting | When is a point a support vector? |
|---|---|
| Hard-margin | $y_i(\mathbf{w}\cdot\mathbf{x}_i + b) = 1$ exactly |
| Soft-margin ($\alpha_i < C$) | On the margin, correctly classified |
| Soft-margin ($\alpha_i = C$) | Inside the margin or misclassified |

## The Math

### Primal SVM (Soft-Margin)

$$\min_{\mathbf{w}, b, \xi} \frac{1}{2}\|\mathbf{w}\|^2 + C\sum_{i=1}^{n}\xi_i$$

subject to: $y_i(\mathbf{w}\cdot\mathbf{x}_i + b) \geq 1 - \xi_i$, $\xi_i \geq 0$

### Dual Formulation

$$\max_{\boldsymbol{\alpha}} \sum_{i=1}^{n}\alpha_i - \frac{1}{2}\sum_{i,j}\alpha_i\alpha_j y_i y_j (\mathbf{x}_i\cdot\mathbf{x}_j)$$

subject to: $0 \leq \alpha_i \leq C$, $\sum_i \alpha_i y_i = 0$

### KKT Conditions

From the KKT complementarity conditions:
$$\alpha_i \big[y_i(\mathbf{w}\cdot\mathbf{x}_i + b) - 1 + \xi_i\big] = 0$$

This means: **if $\alpha_i > 0$**, then $y_i(\mathbf{w}\cdot\mathbf{x}_i + b) = 1 - \xi_i$ (the constraint is tight — the point is a support vector).

### Prediction using only SVs

$$f(\mathbf{x}) = \text{sign}\left(\sum_{i \in \text{SVs}} \alpha_i y_i (\mathbf{x}_i \cdot \mathbf{x}) + b\right)$$

This sum has at most `|SVs|` terms — regardless of the total training set size!

## Setup & Dataset Creation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')

# ── Reproducible 2-D spam dataset with partial overlap ──────────────────────
# Feature 1: "spam keyword density" (0-1 scale)
# Feature 2: "suspicious link ratio"  (0-1 scale)
# Class  +1 = spam,  -1 = legitimate

n = 300

# Spam cluster (upper-right, but with overlap towards center)
spam_kw  = np.random.normal(loc=0.65, scale=0.18, size=n//2)
spam_lnk = np.random.normal(loc=0.60, scale=0.18, size=n//2)

# Legitimate cluster (lower-left, with overlap towards center)
legit_kw  = np.random.normal(loc=0.35, scale=0.18, size=n//2)
legit_lnk = np.random.normal(loc=0.40, scale=0.18, size=n//2)

X = np.vstack([
    np.column_stack([spam_kw,  spam_lnk]),
    np.column_stack([legit_kw, legit_lnk])
])
y = np.array([1]*( n//2) + [-1]*(n//2))

# Clip to [0,1] for realism
X = np.clip(X, 0, 1)

print(f"Dataset: {X.shape[0]} emails, {X.shape[1]} features")
print(f"Spam: {(y==1).sum()}  |  Legitimate: {(y==-1).sum()}")
print(f"\nFeature ranges:")
print(f"  Keyword density : [{X[:,0].min():.2f}, {X[:,0].max():.2f}]")
print(f"  Link ratio       : [{X[:,1].min():.2f}, {X[:,1].max():.2f}]")

## Section 1: How C Controls the Number of Support Vectors

In [ ]:
# ── Helper: plot decision boundary + highlight SVs ────────────────────────
def plot_svm_with_svs(ax, model, X, y, title, C_val):
    """Plot scatter + decision boundary + margin + highlighted SVs."""
    # Background decision-region grid
    h = 0.005
    x_min, x_max = X[:,0].min()-0.05, X[:,0].max()+0.05
    y_min, y_max = X[:,1].min()-0.05, X[:,1].max()+0.05
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=[-999, 0, 999],
                alpha=0.12, colors=['#4C72B0','#DD8452'])
    ax.contour(xx, yy, Z, levels=[-1, 0, 1],
               colors=['#4C72B0','black','#DD8452'],
               linewidths=[1.2, 2.0, 1.2],
               linestyles=['--', '-', '--'])

    # All training points
    ax.scatter(X[y==1,0],  X[y==1,1],  c='#DD8452', edgecolors='k',
               s=25, linewidths=0.5, label='Spam', alpha=0.7, zorder=3)
    ax.scatter(X[y==-1,0], X[y==-1,1], c='#4C72B0', edgecolors='k',
               s=25, linewidths=0.5, label='Legitimate', alpha=0.7, zorder=3)

    # Support vectors — highlighted with a larger ring
    sv = model.support_vectors_
    ax.scatter(sv[:,0], sv[:,1], s=120, facecolors='none',
               edgecolors='red', linewidths=1.8, zorder=4, label='Support Vectors')

    n_sv_spam  = model.n_support_[1] if len(model.classes_) > 1 else model.n_support_[0]
    n_sv_legit = model.n_support_[0]
    ax.set_title(f"{title}\nSVs: {model.n_support_.sum()} "
                 f"(spam={n_sv_spam}, legit={n_sv_legit})", fontsize=11)
    ax.set_xlabel('Keyword Density')
    ax.set_ylabel('Link Ratio')
    ax.legend(fontsize=8, loc='upper left')


# ── Train 3 SVMs with different C values ─────────────────────────────────
C_values = [0.1, 1.0, 10.0]
models = {}

print(f"{'C':>6} | {'Total SVs':>10} | {'Spam SVs':>10} | {'Legit SVs':>10} | {'SV Fraction':>12}")
print('-' * 58)
for C in C_values:
    clf = SVC(kernel='linear', C=C, random_state=42)
    clf.fit(X, y)
    models[C] = clf
    total = clf.n_support_.sum()
    print(f"{C:>6.1f} | {total:>10d} | {clf.n_support_[1]:>10d} | "
          f"{clf.n_support_[0]:>10d} | {total/len(y):>11.1%}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, C in zip(axes, C_values):
    plot_svm_with_svs(ax, models[C], X, y, f'Linear SVM   C = {C}', C)

fig.suptitle('Effect of C on Support Vectors & Decision Boundary',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nObservation: larger C → more SVs (tighter fit to training data)")
print("             smaller C → fewer SVs (wider margin, simpler boundary)")

## Section 2: The Deletion Test — Non-SVs Don't Matter

In [ ]:
# ── Deletion test for C = 1.0 ────────────────────────────────────────────
clf_full = SVC(kernel='linear', C=1.0, random_state=42)
clf_full.fit(X, y)

# Keep only the support vectors
sv_indices = clf_full.support_
X_sv_only  = X[sv_indices]
y_sv_only  = y[sv_indices]

clf_sv_only = SVC(kernel='linear', C=1.0, random_state=42)
clf_sv_only.fit(X_sv_only, y_sv_only)

# Compare decision-function parameters
w_full   = clf_full.coef_[0]
b_full   = clf_full.intercept_[0]
w_svonly = clf_sv_only.coef_[0]
b_svonly = clf_sv_only.intercept_[0]

print("=== Deletion Test ===")
print(f"Training set size  : {len(X)} → {len(X_sv_only)} (only SVs)")
print(f"\nFull model   : w = {w_full},   b = {b_full:.6f}")
print(f"SV-only model: w = {w_svonly}, b = {b_svonly:.6f}")
print(f"\n|Δw|  = {np.linalg.norm(w_full - w_svonly):.2e}")
print(f"|Δb|  = {abs(b_full - b_svonly):.2e}")

# Verify predictions are identical
preds_full   = clf_full.predict(X)
preds_svonly = clf_sv_only.predict(X)
print(f"\nPredictions identical: {np.all(preds_full == preds_svonly)}")
print(f"  Disagreements: {(preds_full != preds_svonly).sum()} out of {len(X)}")

# Visual comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
plot_svm_with_svs(ax1, clf_full,   X, y, 'Full Dataset (300 emails)', 1.0)
plot_svm_with_svs(ax2, clf_sv_only, X_sv_only, y_sv_only,
                  f'Only SVs ({len(X_sv_only)} emails)', 1.0)
fig.suptitle('Deletion Test: Removing Non-SVs Does NOT Change the Boundary',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 3: SV Sensitivity — Moving SVs vs Non-SVs

In [ ]:
# ── Pick one SV and one non-SV, perturb each by 0.1 units ────────────────
np.random.seed(42)
clf_ref = SVC(kernel='linear', C=1.0, random_state=42)
clf_ref.fit(X, y)

sv_idx     = clf_ref.support_[0]           # first support vector
non_sv_mask = np.ones(len(X), dtype=bool)
non_sv_mask[clf_ref.support_] = False
non_sv_idx  = np.where(non_sv_mask)[0][0]  # first non-SV

delta = np.array([0.07, 0.07])             # perturbation vector

# Perturb SV
X_perturb_sv            = X.copy()
X_perturb_sv[sv_idx]   += delta
clf_perturb_sv = SVC(kernel='linear', C=1.0, random_state=42)
clf_perturb_sv.fit(X_perturb_sv, y)

# Perturb non-SV
X_perturb_nonsv             = X.copy()
X_perturb_nonsv[non_sv_idx] += delta
clf_perturb_nonsv = SVC(kernel='linear', C=1.0, random_state=42)
clf_perturb_nonsv.fit(X_perturb_nonsv, y)

w_ref      = clf_ref.coef_[0]
w_sv_pert  = clf_perturb_sv.coef_[0]
w_nsv_pert = clf_perturb_nonsv.coef_[0]

print("=== SV Sensitivity Test ===")
print(f"Perturbation vector: Δ = {delta}  (|Δ| = {np.linalg.norm(delta):.4f})")
print(f"\nPoint perturbed          | |Δw|       | |Δb|")
print(f"{'─'*55}")
print(f"Support vector #{sv_idx:<4d}    | "
      f"{np.linalg.norm(w_ref - w_sv_pert):.4f}     | "
      f"{abs(clf_ref.intercept_[0] - clf_perturb_sv.intercept_[0]):.4f}")
print(f"Non-SV       #{non_sv_idx:<4d}    | "
      f"{np.linalg.norm(w_ref - w_nsv_pert):.6f}   | "
      f"{abs(clf_ref.intercept_[0] - clf_perturb_nonsv.intercept_[0]):.6f}")

# ── Visualize the shifts ──────────────────────────────────────────────────
def get_boundary_line(model, x_range):
    """Return x, y coordinates of the decision boundary line."""
    w = model.coef_[0]
    b = model.intercept_[0]
    # w[0]*x + w[1]*y + b = 0  →  y = -(w[0]*x + b) / w[1]
    x_vals = np.array(x_range)
    y_vals = -(w[0]*x_vals + b) / w[1]
    return x_vals, y_vals

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (pert_label, clf_pert, pert_idx, X_pert) in zip(
    axes,
    [('Support Vector', clf_perturb_sv,    sv_idx,     X_perturb_sv),
     ('Non-SV',         clf_perturb_nonsv, non_sv_idx, X_perturb_nonsv)]
):
    ax.scatter(X[y==1,0],  X[y==1,1],  c='#DD8452', s=20, alpha=0.5, label='Spam')
    ax.scatter(X[y==-1,0], X[y==-1,1], c='#4C72B0', s=20, alpha=0.5, label='Legit')

    x_range = [X[:,0].min()-0.05, X[:,0].max()+0.05]
    xb, yb = get_boundary_line(clf_ref,  x_range)
    xp, yp = get_boundary_line(clf_pert, x_range)
    ax.plot(xb, yb, 'k-',  lw=2.5, label='Original boundary')
    ax.plot(xp, yp, 'r--', lw=2.5, label='After perturbation')

    # Original and perturbed position of the moved point
    ax.scatter(*X[pert_idx], s=180, c='yellow', edgecolors='black',
               zorder=6, label=f'Original {pert_label}')
    ax.scatter(*X_pert[pert_idx], s=180, c='lime', edgecolors='black',
               zorder=6, marker='^', label=f'Moved {pert_label}')
    ax.annotate('', xy=X_pert[pert_idx], xytext=X[pert_idx],
                arrowprops=dict(arrowstyle='->', color='purple', lw=2))

    dw = np.linalg.norm(clf_ref.coef_[0] - clf_pert.coef_[0])
    ax.set_title(f'Perturbing a {pert_label}\n|Δw| = {dw:.4f}', fontsize=11)
    ax.set_xlabel('Keyword Density')
    ax.set_ylabel('Link Ratio')
    ax.set_xlim(x_range)
    ax.set_ylim(X[:,1].min()-0.05, X[:,1].max()+0.05)
    ax.legend(fontsize=8)

fig.suptitle('SV Sensitivity: Moving a Support Vector Shifts the Boundary;\n'
             'Moving a Non-SV Has No Effect', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 4: SV Fraction vs Training Set Size

In [ ]:
# ── Generate a large pool, subsample at different n, measure SV fraction ──
np.random.seed(42)
N_MAX = 2500
kw_spam  = np.random.normal(0.65, 0.18, N_MAX//2)
lnk_spam = np.random.normal(0.60, 0.18, N_MAX//2)
kw_leg   = np.random.normal(0.35, 0.18, N_MAX//2)
lnk_leg  = np.random.normal(0.40, 0.18, N_MAX//2)

X_big = np.clip(np.vstack([
    np.column_stack([kw_spam, lnk_spam]),
    np.column_stack([kw_leg,  lnk_leg])
]), 0, 1)
y_big = np.array([1]*(N_MAX//2) + [-1]*(N_MAX//2))

sample_sizes = [100, 200, 400, 600, 800, 1000, 1500, 2000, 2500]
results = []

for n in sample_sizes:
    idx = np.random.choice(len(X_big), size=n, replace=False)
    Xi, yi = X_big[idx], y_big[idx]
    clf = SVC(kernel='linear', C=1.0, random_state=42)
    clf.fit(Xi, yi)
    total_sv = clf.n_support_.sum()
    results.append({'n': n, 'n_sv': total_sv, 'fraction': total_sv/n})

df_sv = pd.DataFrame(results)
print(df_sv.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(df_sv['n'], df_sv['n_sv'], 'o-', color='#4C72B0', lw=2, ms=7)
ax1.set_xlabel('Training Set Size (n)', fontsize=12)
ax1.set_ylabel('Number of Support Vectors', fontsize=12)
ax1.set_title('Absolute SV Count grows sub-linearly with n', fontsize=11)

ax2.plot(df_sv['n'], df_sv['fraction']*100, 's-', color='#DD8452', lw=2, ms=7)
ax2.set_xlabel('Training Set Size (n)', fontsize=12)
ax2.set_ylabel('Support Vectors (% of training set)', fontsize=12)
ax2.set_title('SV Fraction converges as n grows', fontsize=11)
ax2.axhline(df_sv['fraction'].iloc[-1]*100, color='gray',
            ls='--', label=f"Converges ≈ {df_sv['fraction'].iloc[-1]*100:.1f}%")
ax2.legend()

fig.suptitle('How Many Support Vectors Do We Get as the Dataset Grows?',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nKey insight: as n grows, the SV fraction converges to a constant.")
print("This means SVM prediction cost is O(n_sv) — not O(n) — an efficiency advantage.")

## Section 5: Support Vectors Always Span Both Classes

In [ ]:
# ── Class imbalance and SV class distribution ─────────────────────────────
imbalance_ratios = [0.1, 0.2, 0.3, 0.5]  # fraction of spam
print(f"{'Spam fraction':>14} | {'n_spam':>7} | {'n_legit':>8} | "
      f"{'SV spam':>8} | {'SV legit':>9} | {'Total SVs':>10}")
print('-' * 68)

for ratio in imbalance_ratios:
    np.random.seed(42)
    n_spam  = int(300 * ratio)
    n_legit = 300 - n_spam
    Xs = np.clip(np.column_stack([
        np.random.normal(0.65, 0.18, n_spam),
        np.random.normal(0.60, 0.18, n_spam)
    ]), 0, 1)
    Xl = np.clip(np.column_stack([
        np.random.normal(0.35, 0.18, n_legit),
        np.random.normal(0.40, 0.18, n_legit)
    ]), 0, 1)
    Xi = np.vstack([Xs, Xl])
    yi = np.array([1]*n_spam + [-1]*n_legit)

    clf = SVC(kernel='linear', C=1.0, random_state=42)
    clf.fit(Xi, yi)
    sv_spam  = clf.n_support_[1]
    sv_legit = clf.n_support_[0]
    print(f"{ratio:>14.0%} | {n_spam:>7d} | {n_legit:>8d} | "
          f"{sv_spam:>8d} | {sv_legit:>9d} | {sv_spam+sv_legit:>10d}")

print("\nConclusion: SVs are always found in BOTH classes — they define the boundary from each side.")

## Section 6: Manual Prediction Using Only Support Vectors

We implement the dual prediction formula directly:

$$f(\mathbf{x}) = \text{sign}\!\left(\sum_{i \in \text{SVs}} \alpha_i y_i (\mathbf{x}_i \cdot \mathbf{x}) + b\right)$$

In [ ]:
# ── Train the reference model ─────────────────────────────────────────────
np.random.seed(42)
clf = SVC(kernel='linear', C=1.0, random_state=42)
clf.fit(X, y)

# Sklearn provides dual_coef_ = α_i * y_i  for each SV
alpha_y  = clf.dual_coef_[0]        # shape: (n_sv,)
svs      = clf.support_vectors_     # shape: (n_sv, 2)
b        = clf.intercept_[0]        # scalar

print("Model internals:")
print(f"  Number of SVs : {len(svs)}")
print(f"  dual_coef_    : shape {alpha_y.shape}, range [{alpha_y.min():.3f}, {alpha_y.max():.3f}]")
print(f"  intercept_    : {b:.6f}")

def predict_manual(X_test, svs, alpha_y, b):
    """Dual prediction: f(x) = sign(Σ α_i y_i (sv_i · x) + b)"""
    # dot products: (n_test, n_sv)
    dots = X_test @ svs.T                   # each row: [sv1·x, sv2·x, ...]
    decision_values = dots @ alpha_y + b    # (n_test,)
    return np.sign(decision_values).astype(int)

# Compare on the full training set
preds_manual  = predict_manual(X, svs, alpha_y, b)
preds_sklearn = clf.predict(X)

print(f"\nManual vs sklearn predictions on {len(X)} samples:")
print(f"  Perfect agreement : {np.all(preds_manual == preds_sklearn)}")
print(f"  Disagreements     : {(preds_manual != preds_sklearn).sum()}")

# Also verify decision function values
manual_df  = (X @ svs.T) @ alpha_y + b
sklearn_df = clf.decision_function(X)
print(f"  Max |Δ decision_function| : {np.abs(manual_df - sklearn_df).max():.2e}")

# Show for a few test emails
print("\nSample predictions (first 8 emails):")
print(f"  {'Keyword':>10} {'Link':>6} | {'Manual':>8} | {'Sklearn':>8}")
for i in range(8):
    print(f"  {X[i,0]:>10.3f} {X[i,1]:>6.3f} | "
          f"{'SPAM' if preds_manual[i]==1 else 'LEGIT':>8} | "
          f"{'SPAM' if preds_sklearn[i]==1 else 'LEGIT':>8}")

## Section 7: Accessing SVM Internals in sklearn

A quick reference for the attributes you'll use most.

In [ ]:
# ── Demonstrate all key SVC attributes ───────────────────────────────────
clf_demo = SVC(kernel='linear', C=1.0, random_state=42)
clf_demo.fit(X, y)

print("═" * 55)
print(" Key sklearn SVC attributes (C=1.0, linear kernel)")
print("═" * 55)

print("\n► model.support_vectors_")
print(f"  Shape       : {clf_demo.support_vectors_.shape}")
print(f"  First 3 SVs : \n{clf_demo.support_vectors_[:3]}")

print("\n► model.support_")
print(f"  Shape       : {clf_demo.support_.shape}")
print(f"  First 5 SV indices: {clf_demo.support_[:5]}")

print("\n► model.n_support_")
print(f"  Value       : {clf_demo.n_support_}  (legit SVs, spam SVs)")

print("\n► model.dual_coef_")
print(f"  Shape       : {clf_demo.dual_coef_.shape}  (= α_i * y_i per SV)")
print(f"  Range       : [{clf_demo.dual_coef_.min():.4f}, {clf_demo.dual_coef_.max():.4f}]")

print("\n► model.coef_  (linear kernel only)")
print(f"  w = {clf_demo.coef_[0]}")

print("\n► Verify w = Σ α_i y_i x_i")
w_manual = (clf_demo.dual_coef_[0][:, None] * clf_demo.support_vectors_).sum(axis=0)
w_sklearn = clf_demo.coef_[0]
print(f"  w (manual)  = {w_manual}")
print(f"  w (sklearn) = {w_sklearn}")
print(f"  |Δw|        = {np.linalg.norm(w_manual - w_sklearn):.2e}")

print("\n► model.intercept_")
print(f"  b = {clf_demo.intercept_[0]:.6f}")

## Section 8: Summary Visualization — Everything Together

In [ ]:
# ── Comprehensive 4-panel summary figure ─────────────────────────────────
fig = plt.figure(figsize=(15, 11))
gs  = fig.add_gridspec(2, 2, hspace=0.4, wspace=0.35)

# --- Panel 1: SVM with C=1 and all SV details ----------------------------
ax1 = fig.add_subplot(gs[0, 0])
clf_p = SVC(kernel='linear', C=1.0, random_state=42)
clf_p.fit(X, y)
plot_svm_with_svs(ax1, clf_p, X, y, 'Support Vectors (C=1.0)', 1.0)

# --- Panel 2: Dual coefficients for each SV ------------------------------
ax2 = fig.add_subplot(gs[0, 1])
dc = clf_p.dual_coef_[0]  # α_i * y_i
sv_y_labels = y[clf_p.support_]
colors_dc = ['#DD8452' if yi == 1 else '#4C72B0' for yi in sv_y_labels]
ax2.bar(range(len(dc)), dc, color=colors_dc, edgecolor='black', linewidth=0.5)
ax2.axhline(0, color='black', lw=0.8)
ax2.set_xlabel('Support Vector Index', fontsize=11)
ax2.set_ylabel('Dual Coefficient (αᵢ · yᵢ)', fontsize=11)
ax2.set_title('Dual Coefficients of Each Support Vector', fontsize=11)
spam_patch  = mpatches.Patch(color='#DD8452', label='Spam SV (αᵢyᵢ > 0)')
legit_patch = mpatches.Patch(color='#4C72B0', label='Legit SV (αᵢyᵢ < 0)')
ax2.legend(handles=[spam_patch, legit_patch], fontsize=9)

# --- Panel 3: SV fraction vs C -------------------------------------------
ax3 = fig.add_subplot(gs[1, 0])
C_range = np.logspace(-2, 2, 25)
sv_fracs = []
for Cval in C_range:
    m = SVC(kernel='linear', C=Cval, random_state=42).fit(X, y)
    sv_fracs.append(m.n_support_.sum() / len(y))
ax3.semilogx(C_range, [f*100 for f in sv_fracs], 'o-', color='#2ca02c', lw=2, ms=5)
ax3.set_xlabel('Regularization C (log scale)', fontsize=11)
ax3.set_ylabel('Support Vectors (% of training set)', fontsize=11)
ax3.set_title('SV Fraction vs C: More C → More SVs', fontsize=11)
ax3.axvline(1.0, color='red', ls='--', alpha=0.6, label='C=1')
ax3.legend(fontsize=9)

# --- Panel 4: Distance of each point from boundary -----------------------
ax4 = fig.add_subplot(gs[1, 1])
df_vals  = clf_p.decision_function(X)
is_sv    = np.zeros(len(X), dtype=bool)
is_sv[clf_p.support_] = True
spam_mask = y == 1

bins = np.linspace(-3, 3, 35)
ax4.hist(df_vals[spam_mask  & ~is_sv], bins=bins, color='#DD8452', alpha=0.5, label='Spam (non-SV)')
ax4.hist(df_vals[~spam_mask & ~is_sv], bins=bins, color='#4C72B0', alpha=0.5, label='Legit (non-SV)')
ax4.hist(df_vals[spam_mask  &  is_sv], bins=bins, color='#DD8452', alpha=0.9,
         edgecolor='red', linewidth=1.5, label='Spam SV')
ax4.hist(df_vals[~spam_mask &  is_sv], bins=bins, color='#4C72B0', alpha=0.9,
         edgecolor='red', linewidth=1.5, label='Legit SV')
ax4.axvline(-1, color='blue',  ls='--', lw=1.5, alpha=0.7)
ax4.axvline( 0, color='black', ls='-',  lw=1.5)
ax4.axvline( 1, color='orange',ls='--', lw=1.5, alpha=0.7)
ax4.set_xlabel('Decision Function Value f(x)', fontsize=11)
ax4.set_ylabel('Count', fontsize=11)
ax4.set_title('SVs Cluster Near the Margin (|f(x)| ≈ 1)', fontsize=11)
ax4.legend(fontsize=8)

fig.suptitle('Support Vectors: Complete Picture',
             fontsize=15, fontweight='bold', y=1.01)
plt.show()

## Self-Check Questions

Test your understanding before moving on. Think through each answer, then expand the solution.

---

### Q1
You train an SVM on 10,000 emails. Only 42 are support vectors. What is the **minimum number of training emails** needed to retrain and get the **identical model**?

<details>
<summary>Show Answer</summary>

**42 emails — just the support vectors.**

Because the model (w, b) is defined *entirely* by the dual coefficients and positions of the support vectors:

$$\mathbf{w} = \sum_{i \in \text{SVs}} \alpha_i y_i \mathbf{x}_i$$

All 9,958 non-support vectors have $\alpha_i = 0$, so they contribute nothing to `w` or `b`. Retraining on only the 42 SVs (with the same `C`) produces the identical hyperplane. This is exactly what the deletion test in Section 2 demonstrated — we got `|Δw| = 0` and `|Δb| = 0`.

**Practical implication:** You can compress your SVM model by storing only support vectors — a huge memory saving!

</details>

---

### Q2
As C increases from 0.01 to 100, the number of support vectors changes. Does it **increase or decrease**? Why?

<details>
<summary>Show Answer</summary>

**It increases (generally) as C increases.**

Intuition:
- **Small C** → large regularization → wide margin is prioritized over correctly classifying all points → the model allows many points to be inside the margin or misclassified → but only borderline points have $\alpha_i > 0$ → **fewer SVs**.
- **Large C** → small regularization → the model tries to classify every point correctly → the margin shrinks to fit tightly around the data → more points end up on or inside the margin → **more SVs**.

The soft-margin condition: $\alpha_i = C$ for misclassified/margin-violating points. At high C, even small violations (points barely inside the margin) are penalized, making many points become SVs with $\alpha_i = C$.

**Warning:** This relationship is not always strictly monotone, but the general trend holds.

</details>

---

### Q3
You add 500 new training emails that are **far from the decision boundary**. How many of these will become support vectors? What happens to the existing SVs?

<details>
<summary>Show Answer</summary>

**Approximately 0 of the 500 new emails become support vectors.** The existing SVs are largely unchanged.

Here's why:
- Support vectors are defined as the points *closest* to the decision boundary — the ones with $\alpha_i > 0$ (which corresponds to $y_i f(\mathbf{x}_i) \leq 1$).
- Points far from the boundary satisfy $y_i f(\mathbf{x}_i) \gg 1$, which means KKT forces $\alpha_i = 0$ for them.
- Since $\alpha_i = 0$, these points do not contribute to `w` or `b`, so they are not SVs.

**What happens to existing SVs?** Adding far-away points doesn't move the boundary (because the far points don't become SVs), so the original SVs remain SVs with the same or very similar $\alpha_i$ values. The model is robust to adding irrelevant data far from the boundary.

**Contrast:** If you add 500 emails *close to the boundary* (in the overlap region), many WILL become SVs and the boundary could shift.

</details>

## Key Takeaways

| Concept | Summary |
|---|---|
| **What are SVs?** | Training points with $\alpha_i > 0$; they lie on or inside the margin |
| **Why do they matter?** | They are the ONLY points that determine `w` and `b` |
| **Effect of C** | Larger C → more SVs; Smaller C → fewer SVs |
| **Robustness** | Moving/removing non-SVs has zero effect on the model |
| **Prediction** | $f(\mathbf{x}) = \sum_{\text{SVs}} \alpha_i y_i (\mathbf{x}_i \cdot \mathbf{x}) + b$ |
| **Compression** | Store only SVs — same model, tiny storage |

---

**Next:** Notebook 06 — The Kernel Trick: what happens when emails can't be separated by a straight line?